# Volume 3, Chapter 12: The Probabilistic Method (Python companion)

Exercises first, solutions at the end. We compare random and algebraic Ramsey colourings, run the Moser–Tardos algorithm that makes the local lemma constructive, watch thresholds sharpen in random graphs, measure the giant component against the branching process, and turn random cuts and assignments into deterministic algorithms.

*Before each solution, predict what the output will look like.*

In [ ]:
import math, itertools
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(12)

## Exercises

**P12.1 (★) Random against algebraic.** $R(4)=18$: some colouring of $K_{17}$ has no monochromatic $K_4$. Compute the expected number of monochromatic $K_4$ in a random colouring of $K_{17}$, and count them in a few random colourings. Then colour the edge $\{i,j\}$ red when $i-j$ is a non-zero square modulo $17$ (the *Paley graph*) and check that there is no monochromatic $K_4$. What does Theorem 12.2 give for $k=4$?

**P12.2 (★★) Moser–Tardos.** Build a $k$-CNF formula with $k=8$ in which every variable appears in exactly $4$ clauses, so that each clause meets at most $28$ others and Corollary 12.8 applies ($e\cdot29<2^8$). Run the resampling algorithm for $n=10^3,10^4,10^5$ variables and count the resamplings. Then try it on random $3$-CNF formulas with $\frac mn=2.5$ and $4.2$ clauses per variable, far outside the local lemma.

**P12.3 (★) Sharp and coarse thresholds.** Estimate $P(T>0)$ for the number $T$ of triangles in $G(n,\frac cn)$, for $n=50,100,200$ and $c$ from $0$ to $3$, and compare with $1-e^{-c^3/6}$ (Theorem 12.6). Then estimate $P(\text{connected})$ for $G(n,\frac{\ln n+c}n)$ with $n=2000$ and compare with $e^{-e^{-c}}$ (Exercise 12.6).

**P12.4 (★★) The giant component.** For $n=10^5$, compute the sizes of the two largest components of $G(n,\frac cn)$ for $c=0.5,0.9,1,1.1,1.5,2,3$, and compare with $\rho(c)n$ and with the bound $\frac{\ln n}{c-1-\ln c}$ of Theorem 12.9.

**P12.5 (★) Derandomisation.** For random $3$-CNF formulas with $n=200$ variables and $m=1000$ clauses, compare the number of satisfied clauses for a random assignment, for the method of conditional expectations (Theorem 12.10), and for the method followed by a simple local search. Do the same for MAX-CUT on $G(1000,0.01)$.

## Solutions

**P12.1.**

In [ ]:
n = 17
def mono_k4(colour):
    count = 0
    for S in itertools.combinations(range(n), 4):
        c = {colour[i][j] for i, j in itertools.combinations(S, 2)}
        if len(c) == 1: count += 1
    return count
print('expected number in a random colouring:', math.comb(17, 4) * 2**-5)
for _ in range(3):
    col = np.zeros((n, n), dtype=int)
    for i, j in itertools.combinations(range(n), 2):
        col[i][j] = col[j][i] = rng.integers(2)
    print('random colouring:', mono_k4(col), 'monochromatic K4')
squares = {(x * x) % 17 for x in range(1, 17)}
paley = [[int((i - j) % 17 in squares) for j in range(n)] for i in range(n)]
print('Paley colouring:', mono_k4(paley), 'monochromatic K4;  squares mod 17:', sorted(squares))
print('first moment bound for k=4: largest n with C(n,4) 2^-5 < 1 is', max(m for m in range(4, 40) if math.comb(m, 4) * 2**-5 < 1))

A random colouring of $K_{17}$ has about $74$ monochromatic $K_4$: the probabilistic method is hopeless here, and only proves $R(4)>6$. The Paley colouring, built from the squares modulo $17$, has none, and $17$ is the largest possible. The roles are reversed for large $k$: algebraic constructions such as Paley graphs are conjectured to be good Ramsey colourings, but nobody can prove they beat $1.0001^k$, while random colourings provably reach $\sqrt2^{\,k}$. Since $-1$ is a square modulo $17$ ($4^2=16$), the relation is symmetric, which is why $17\equiv1\pmod4$ is needed.

**P12.2.** The formula is built by the configuration model: list each variable $4$ times, shuffle, cut into clauses of $8$, and reshuffle the clauses that contain a repeated variable. The algorithm keeps the list of false clauses up to date through the clauses of each variable.

In [ ]:
def lll_formula(nvar, k=8, occ=4):
    while True:
        slots = np.repeat(np.arange(nvar), occ); rng.shuffle(slots)
        clauses = slots.reshape(-1, k)
        bad = [i for i, c in enumerate(clauses) if len(set(c)) < k]
        tries = 0
        while bad and tries < 1000:
            # swap entries of bad clauses with random positions
            for i in bad:
                for pos in range(k):
                    j = rng.integers(len(clauses)); q = rng.integers(k)
                    clauses[i, pos], clauses[j, q] = clauses[j, q], clauses[i, pos]
            bad = [i for i, c in enumerate(clauses) if len(set(c)) < k]
            tries += 1
        if not bad:
            signs = rng.integers(2, size=clauses.shape).astype(bool)
            return clauses, signs
def moser_tardos(clauses, signs, nvar):
    x = rng.integers(2, size=nvar).astype(bool)
    occ = [[] for _ in range(nvar)]
    for i, c in enumerate(clauses):
        for v in c: occ[v].append(i)
    def false(i):
        return not np.any(x[clauses[i]] == signs[i])
    bad = {i for i in range(len(clauses)) if false(i)}
    resamples = 0
    while bad:
        i = bad.pop()
        x[clauses[i]] = rng.integers(2, size=len(clauses[i])).astype(bool)
        resamples += 1
        for v in clauses[i]:
            for j in occ[v]:
                if false(j): bad.add(j)
                else: bad.discard(j)
    return resamples, x
for nvar in [1000, 10_000, 100_000]:
    cl, sg = lll_formula(nvar)
    res, x = moser_tardos(cl, sg, nvar)
    ok = all(np.any(x[c] == s) for c, s in zip(cl, sg))
    print(f'n = {nvar:6d} variables, {len(cl):6d} clauses: {res:5d} resamplings, satisfied: {ok};  Moser-Tardos bound m/d = {len(cl) / 28:.0f}')

In [ ]:
def random_3cnf(nvar, m):
    cl = np.array([rng.choice(nvar, 3, replace=False) for _ in range(m)])
    return cl, rng.integers(2, size=(m, 3)).astype(bool)
for ratio in [2.5, 4.2]:
    nvar = 2000; m = int(ratio * nvar)
    cl, sg = random_3cnf(nvar, m)
    # cap the work: give up after 200 000 resamplings
    x = rng.integers(2, size=nvar).astype(bool)
    occ = [[] for _ in range(nvar)]
    for i, c in enumerate(cl):
        for v in c: occ[v].append(i)
    false = lambda i: not np.any(x[cl[i]] == sg[i])
    bad = {i for i in range(m) if false(i)}; steps = 0
    while bad and steps < 200_000:
        i = bad.pop(); x[cl[i]] = rng.integers(2, size=3).astype(bool); steps += 1
        for v in cl[i]:
            for j in occ[v]:
                if false(j): bad.add(j)
                else: bad.discard(j)
    print(f'm/n = {ratio}: {"solved" if not bad else "not solved"} after {steps} resamplings, {len(bad)} false clauses left')

The number of resamplings grows linearly with the size of the formula and stays far below the bound $\frac md$: most clauses are true from the start (each is false with probability $2^{-8}$), and a resampling rarely breaks a neighbour. Outside the local lemma the picture changes. Random $3$-CNF formulas are satisfiable w.h.p.\ up to about $4.27$ clauses per variable (a threshold located by Ding, Sly and Sun in 2015, for large $k$), but simple resampling succeeds only well below it: here it needs over a hundred thousand resamplings at ratio $2.5$, and stalls at $4.2$ (it already stalls at $3.5$; pure random walk is known to work only up to about $2.7$), while cleverer algorithms such as WalkSAT with greedy moves, or survey propagation, still succeed near the threshold.

**P12.3.**

In [ ]:
cs = np.linspace(0.2, 3.0, 15)
plt.figure(figsize=(7, 3))
for n in [50, 100, 200]:
    probs = []
    for c in cs:
        hits = 0
        for _ in range(300):
            A = (rng.random((n, n)) < c / n).astype(np.int32)
            A = np.triu(A, 1); A = A + A.T
            hits += np.trace(A @ A @ A) > 0
        probs.append(hits / 300)
    plt.plot(cs, probs, 'o-', ms=3, label=f'n={n}')
plt.plot(cs, 1 - np.exp(-cs**3 / 6), 'k', label='$1-e^{-c^3/6}$')
plt.xlabel('c = np'); plt.ylabel('P(triangle)'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def components(n, edges):
    parent = np.arange(n)
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    roots = np.array([find(a) for a in range(n)])
    return np.bincount(roots, minlength=n)
def gnp_edges(n, p):
    m = rng.binomial(n * (n - 1) // 2, p)
    e = rng.integers(n, size=(int(m * 1.02) + 10, 2))
    e = e[e[:, 0] != e[:, 1]]
    e = np.unique(np.sort(e, axis=1), axis=0)
    rng.shuffle(e)
    return e[:m]
n, reps = 2000, 200
for c in [-1, 0, 1, 2]:
    p = (math.log(n) + c) / n
    conn = sum(components(n, gnp_edges(n, p)).max() == n for _ in range(reps)) / reps
    print(f'c = {c:+d}: P(connected) = {conn:.3f}   e^(-e^(-c)) = {math.exp(-math.exp(-c)):.3f}')

For triangles the curves for different $n$ coincide with $1-e^{-c^3/6}$: the threshold is *coarse*, the probability rising smoothly over a window of $c$ of constant width. For connectivity the window has width $\frac1n$ in $p$ around $\frac{\ln n}n$, which is small compared with $\frac{\ln n}n$: the threshold is *sharp*, and the transition is described by the Gumbel law $e^{-e^{-c}}$ of the coupon collector (Chapter 9), for the same reason: the last vertex to be connected is the last coupon to arrive. Friedgut's theorem (1999) says, roughly, that a threshold is coarse only when the property is determined by a small subgraph, like containing a triangle.

**P12.4.**

In [ ]:
def rho(c):
    if c <= 1: return 0.0
    r = 0.5
    for _ in range(5000): r = 1 - math.exp(-c * r)
    return r
n = 100_000
print('   c    largest    rho(c) n    second    ln n / I(c)')
for c in [0.5, 0.9, 1.0, 1.1, 1.5, 2.0, 3.0]:
    sizes = np.sort(components(n, gnp_edges(n, c / n)))[::-1]
    I = c - 1 - math.log(c) if c != 1 else float('nan')
    bound = math.log(n) / I if c < 1 else float('nan')
    print(f'{c:4.1f}  {sizes[0]:8d}   {rho(c) * n:9.0f}   {sizes[1]:6d}    {bound:8.1f}')

Below $c=1$ the largest component is small and stays below the bound $\frac{\ln n}I$ of Theorem 12.9: $27$ against $60$ at $c=0.5$, and $316$ against $2148$ at $c=0.9$, where the bound is far from tight because $I$ is tiny. At $c=1$ the largest component has a few thousand vertices, of the order of $n^{2/3}\approx2154$ (the constant fluctuates a lot from one graph to the next, and the second largest is of the same size). Above $1$, one component holds a fraction $\rho(c)$ of the vertices, within a few percent at $c=1.1$ and a fraction of a percent beyond, while the second largest is small again and shrinks as $c$ grows (a few hundred vertices at $c=1.1$, five at $c=3$): the giant is unique. Near $c=1$ everything is slow, because the sizes of the small components are of order $\frac{\ln n}{I}$ there too, with $I$ close to $0$. These are the numbers of Exercise 3.8: for $c=2$, $\rho=0.797$.

**P12.5.**

In [ ]:
def satisfied(cl, sg, x):
    return int(np.sum(np.any(x[cl] == sg, axis=1)))
def cond_exp_assign(cl, sg, nvar):
    x = np.zeros(nvar, dtype=bool); fixed = np.zeros(nvar, dtype=bool)
    def W():
        val = x[cl] == sg; fx = fixed[cl]
        sat = np.any(val & fx, axis=1)
        free = (~fx).sum(axis=1)
        return np.sum(np.where(sat, 1.0, 1.0 - 0.5**free))
    for v in range(nvar):
        fixed[v] = True
        x[v] = True; w1 = W()
        x[v] = False; w0 = W()
        x[v] = w1 >= w0
    return x
def local_search(cl, sg, x):
    x = x.copy(); best = satisfied(cl, sg, x); improved = True
    while improved:
        improved = False
        for v in range(len(x)):
            x[v] = ~x[v]; s = satisfied(cl, sg, x)
            if s > best: best = s; improved = True
            else: x[v] = ~x[v]
    return x
nvar, m = 200, 1000
cl, sg = random_3cnf(nvar, m)
xr = rng.integers(2, size=nvar).astype(bool)
xc = cond_exp_assign(cl, sg, nvar)
print('MAX-3SAT, m = 1000:  7/8 m =', 7 * m / 8, ' random:', satisfied(cl, sg, xr), ' conditional expectations:', satisfied(cl, sg, xc),
      ' + local search:', satisfied(cl, sg, local_search(cl, sg, xc)))
n, p = 1000, 0.01
A = np.triu(rng.random((n, n)) < p, 1); E = np.argwhere(A)
side = np.zeros(n, dtype=int); placed = np.zeros(n, dtype=bool)
adj = [[] for _ in range(n)]
for a, b in E: adj[a].append(b); adj[b].append(a)
for v in range(n):
    left = sum(1 for u in adj[v] if placed[u] and side[u] == 0)
    right = sum(1 for u in adj[v] if placed[u] and side[u] == 1)
    side[v] = 1 if left >= right else 0          # join the side that cuts more placed neighbours
    placed[v] = True
cut = lambda s: int(np.sum(s[E[:, 0]] != s[E[:, 1]]))
print('MAX-CUT: m =', len(E), ' m/2 =', len(E) / 2, ' random:', cut(rng.integers(2, size=n)), ' greedy (derandomised):', cut(side))

The derandomised assignment beats $\frac78m$ comfortably, because the conditional expectations also exploit the luck of earlier choices, and local search adds a little more. Random $3$-CNF formulas with $5$ clauses per variable are unsatisfiable w.h.p., but about $97\%$ to $98\%$ of the clauses can be satisfied. The greedy cut, the derandomisation of Proposition 12.3, beats $\frac m2$ by about a third: it cuts about $67\%$ of the edges, against $50\%$ for a random cut, which is typical of sparse random graphs, where the maximum cut is well above $\frac m2$. For MAX-CUT, Goemans and Williamson (1995) found a randomised rounding of a semidefinite program that guarantees a fraction $0.878$ of the optimum, and under the Unique Games Conjecture nothing better is possible in polynomial time: random choices, made cleverly, are again optimal.